<a href="https://colab.research.google.com/github/MahaThafar/Diabetic-Retinopathy-CNN-Transformer-Comparison/blob/main/mobilenetv2_DR_5Seeds.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Five-seed DR classification experiment - Mobilenetv2 Model

This notebook runs one architecture with five predefined random seeds and saves every completed run immediately. It adds:

- five repeated runs and **mean ± sample standard deviation**;
- **Quadratic Weighted Kappa (QWK)**;
- macro and weighted one-vs-rest multiclass AUC;
- hierarchical stratified-bootstrap 95% confidence intervals;
- per-class confidence intervals;
- batch-size-1 inference latency under identical conditions;
- saved predictions/probabilities for later McNemar and paired-bootstrap comparisons;
- cross-model Friedman and Wilcoxon–Holm analysis cells for use after all six models are completed.

The checkpoint is selected by **validation macro F1-score**, matching the imbalanced five-class objective. Do not describe the selected epoch as the epoch of minimum validation loss unless you change the selection criterion.

In [ ]:
!pip install -q timm kagglehub scipy statsmodels
!pip install timm -q
!pip install timm kagglehub -q

In [ ]:
import os
import gc
import glob
import json
import math
import random
import platform
import subprocess
import time
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms

import timm
import kagglehub

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    cohen_kappa_score,
)
from sklearn.preprocessing import label_binarize
from scipy.stats import t, friedmanchisquare, wilcoxon
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.contingency_tables import mcnemar

print('PyTorch:', torch.__version__)
print('TIMM:', timm.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))

print('CUDA available:', torch.cuda.is_available())


PyTorch: 2.11.0+cu128
TIMM: 1.0.28
Device: cuda
GPU: NVIDIA A100-SXM4-80GB
CUDA available: True


In [ ]:
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Configuration

Change only `MODEL_NAME`, `SAVE_NAME`, and—only when memory requires it—`BATCH_SIZE`. Use the same five seeds and all other settings for every architecture.

In [ ]:
class CFG:
    # Current model
    MODEL_NAME = 'mobilenetv2_100'
    SAVE_NAME = 'mobilenetv2'

    # Other exact TIMM identifiers:
    # 'efficientnet_b0'
    # 'mobilenetv2_100'
    # 'vit_base_patch16_224'
    # 'swin_tiny_patch4_window7_224'
    # 'swin_base_patch4_window7_224'

    NUM_CLASSES = 5
    CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative DR']
    IMG_SIZE = 224

    BATCH_SIZE = 32       # use 16 only if the selected architecture runs out of memory
    NUM_EPOCHS = 20
    LEARNING_RATE = 5e-5
    WEIGHT_DECAY = 1e-4
    PATIENCE = 5
    NUM_WORKERS = 2

    SEEDS = [42, 123, 2024, 3407, 7777]
    SELECTION_METRIC = 'f1_macro'

    N_BOOTSTRAP = 2000
    CONFIDENCE_LEVEL = 0.95

    LATENCY_WARMUP = 30
    LATENCY_REPEATS = 5

    # Set to a Google Drive folder if desired, e.g.
    # '/content/drive/MyDrive/DR_revision_results'
   # OUTPUT_ROOT = '/content/DR_revision_results'
    OUTPUT_ROOT = "/content/drive/MyDrive/DR_revision_results"

OUTPUT_DIR = Path(CFG.OUTPUT_ROOT) / CFG.SAVE_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Output:', OUTPUT_DIR)

Device: cuda
Output: /content/drive/MyDrive/DR_revision_results/mobilenetv2


### Optional: save directly to Google Drive

Run this cell only if you want completed seeds to survive a Colab disconnection. Then change `CFG.OUTPUT_ROOT` above to a Drive path and rerun the configuration cell.

## Dataset loading and audit

In [ ]:
DATASET_PATH = kagglehub.dataset_download('mariaherrerot/aptos2019')
print('Dataset path:', DATASET_PATH)

train_csv = os.path.join(DATASET_PATH, 'train_1.csv')
val_csv = os.path.join(DATASET_PATH, 'valid.csv')
test_csv = os.path.join(DATASET_PATH, 'test.csv')

train_img_dir = os.path.join(DATASET_PATH, 'train_images')
val_img_dir = os.path.join(DATASET_PATH, 'val_images')
test_img_dir = os.path.join(DATASET_PATH, 'test_images')

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

REQUIRED_COLUMNS = {'id_code', 'diagnosis'}
for split_name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    missing = REQUIRED_COLUMNS - set(df.columns)
    if missing:
        raise ValueError(f'{split_name} is missing columns: {missing}')
    invalid = sorted(set(df['diagnosis'].unique()) - set(range(CFG.NUM_CLASSES)))
    if invalid:
        raise ValueError(f'{split_name} contains invalid labels: {invalid}')

# Correct class counts for Table 1: generated directly from the CSV files.
split_counts = pd.DataFrame({
    'Train': train_df['diagnosis'].value_counts().reindex(range(CFG.NUM_CLASSES), fill_value=0),
    'Validation': val_df['diagnosis'].value_counts().reindex(range(CFG.NUM_CLASSES), fill_value=0),
    'Test': test_df['diagnosis'].value_counts().reindex(range(CFG.NUM_CLASSES), fill_value=0),
})
split_counts.index = [f'{i} - {name}' for i, name in enumerate(CFG.CLASS_NAMES)]
split_counts['Total'] = split_counts.sum(axis=1)
split_counts.loc['Total'] = split_counts.sum(axis=0)

print(split_counts)
split_counts.to_csv(OUTPUT_DIR / 'verified_dataset_split_counts.csv')

print('\nLabel mapping:')
for idx, name in enumerate(CFG.CLASS_NAMES):
    print(f'{idx} = {name}')

100%|██████████| 8.01G/8.01G [03:26<00:00, 41.7MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/mariaherrerot/aptos2019/versions/3
                      Train  Validation  Test  Total
0 - No DR              1434         172   199   1805
1 - Mild                300          40    30    370
2 - Moderate            808         104    87    999
3 - Severe              154          22    17    193
4 - Proliferative DR    234          28    33    295
Total                  2930         366   366   3662

Label mapping:
0 = No DR
1 = Mild
2 = Moderate
3 = Severe
4 = Proliferative DR


In [ ]:
def find_image_path(image_dir, image_id):
    candidates = [
        os.path.join(image_dir, f'{image_id}.png'),
        os.path.join(image_dir, '**', f'{image_id}.png'),
    ]
    for pattern in candidates:
        matches = glob.glob(pattern, recursive=True)
        if matches:
            return matches[0]
    return None

for df, image_dir in [(train_df, train_img_dir), (val_df, val_img_dir), (test_df, test_img_dir)]:
    df['image_path'] = df['id_code'].apply(lambda x: find_image_path(image_dir, x))

for split_name, df in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    missing_paths = df['image_path'].isna().sum()
    if missing_paths:
        raise FileNotFoundError(f'{split_name}: {missing_paths} images were not resolved.')

print('All image paths were resolved.')

All image paths were resolved.


## Transformations and dataset objects

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((CFG.IMG_SIZE, CFG.IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class RetinalDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = int(row['diagnosis'])
        if self.transform is not None:
            image = self.transform(image)
        return image, label, idx

train_dataset = RetinalDataset(train_df, train_transform)
val_dataset = RetinalDataset(val_df, val_test_transform)
test_dataset = RetinalDataset(test_df, val_test_transform)

## Reproducibility, class weights, and seed-specific loaders

In [ ]:
def set_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed)
    random.seed(worker_seed)

class_counts = (
    train_df['diagnosis']
    .value_counts()
    .reindex(range(CFG.NUM_CLASSES), fill_value=0)
    .sort_index()
    .to_numpy()
)
if np.any(class_counts == 0):
    raise ValueError(f'At least one training class is empty: {class_counts}')

class_weights_np = 1.0 / class_counts.astype(np.float64)
class_weights_np = class_weights_np / class_weights_np.sum() * CFG.NUM_CLASSES
sample_weights_np = train_df['diagnosis'].map(
    {i: class_weights_np[i] for i in range(CFG.NUM_CLASSES)}
).to_numpy(dtype=np.float64)

print('Training counts:', class_counts)
print('Class weights:', class_weights_np)


def create_loaders(seed):
    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights_np, dtype=torch.double),
        num_samples=len(sample_weights_np),
        replacement=True,
        generator=generator,
    )

    common = dict(
        num_workers=CFG.NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
        worker_init_fn=seed_worker,
        persistent_workers=(CFG.NUM_WORKERS > 0),
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=CFG.BATCH_SIZE,
        sampler=sampler,
        generator=generator,
        **common,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        **common,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=CFG.BATCH_SIZE,
        shuffle=False,
        **common,
    )
    return train_loader, val_loader, test_loader

Training counts: [1434  300  808  154  234]
Class weights: [0.21744192 1.03937239 0.38590559 2.0247514  1.3325287 ]


## Metrics

In [ ]:
def calculate_metrics(y_true, y_pred, y_proba=None, num_classes=5):
    y_true = np.asarray(y_true, dtype=int)
    y_pred = np.asarray(y_pred, dtype=int)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average='weighted', zero_division=0
    )

    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': p_macro,
        'recall_macro': r_macro,
        'f1_macro': f1_macro,
        'precision_weighted': p_weighted,
        'recall_weighted': r_weighted,
        'f1_weighted': f1_weighted,
        'qwk': cohen_kappa_score(
            y_true, y_pred, labels=list(range(num_classes)), weights='quadratic'
        ),
    }

    if y_proba is not None:
        y_proba = np.asarray(y_proba, dtype=float)
        y_true_bin = label_binarize(y_true, classes=list(range(num_classes)))
        try:
            metrics['auc_macro_ovr'] = roc_auc_score(
                y_true_bin, y_proba, average='macro', multi_class='ovr'
            )
            metrics['auc_weighted_ovr'] = roc_auc_score(
                y_true_bin, y_proba, average='weighted', multi_class='ovr'
            )
        except ValueError:
            metrics['auc_macro_ovr'] = np.nan
            metrics['auc_weighted_ovr'] = np.nan

    return metrics


def calculate_per_class_metrics(y_true, y_pred):
    p, r, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(CFG.NUM_CLASSES)),
        average=None,
        zero_division=0,
    )
    return pd.DataFrame({
        'class_index': range(CFG.NUM_CLASSES),
        'class_name': CFG.CLASS_NAMES,
        'precision': p,
        'recall': r,
        'f1': f1,
        'support': support,
    })

## Training and validation functions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_labels, all_preds = [], []

    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()

    for images, labels, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()



        running_loss += loss.item() * images.size(0)
        all_labels.extend(labels.detach().cpu().numpy())
        all_preds.extend(logits.argmax(dim=1).detach().cpu().numpy())

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start



    return (
        running_loss / len(loader.dataset),
        calculate_metrics(all_labels, all_preds),
        elapsed,
    )


def evaluate(model, loader, criterion, device, return_proba=False):
    model.eval()
    running_loss = 0.0
    records = []

    if device.type == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()

    with torch.inference_mode():
        for images, labels, indices in loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(images)
            loss = criterion(logits, labels)
            probs = torch.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            running_loss += loss.item() * images.size(0)
            for i in range(images.size(0)):
                rec = {
                    'sample_index': int(indices[i]),
                    'y_true': int(labels[i].item()),
                    'y_pred': int(preds[i].item()),
                }
                if return_proba:
                    for c in range(CFG.NUM_CLASSES):
                        rec[f'prob_{c}'] = float(probs[i, c].item())
                records.append(rec)

    if device.type == 'cuda':
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    pred_df = pd.DataFrame(records).sort_values('sample_index').reset_index(drop=True)
    y_true = pred_df['y_true'].to_numpy()
    y_pred = pred_df['y_pred'].to_numpy()
    y_proba = pred_df[[f'prob_{c}' for c in range(CFG.NUM_CLASSES)]].to_numpy() if return_proba else None
    metrics = calculate_metrics(y_true, y_pred, y_proba)

    return running_loss / len(loader.dataset), metrics, pred_df, elapsed

## Batch-size-1 latency measurement

In [ ]:
def create_latency_loader():
    return DataLoader(
        test_dataset,
        batch_size=1,
        shuffle=False,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )


def measure_batch1_forward_latency(
    model,
    loader,
    device,
    warmup_iterations=30,
    repeats=5,
):
    """Model-forward latency only; excludes file loading and host-to-device transfer."""
    model.eval()
    iterator = iter(loader)
    first_images, _, _ = next(iterator)
    first_images = first_images.to(device)

    with torch.inference_mode():
        for _ in range(warmup_iterations):
            _ = model(first_images)
    if device.type == 'cuda':
        torch.cuda.synchronize()

    latencies_ms = []
    with torch.inference_mode():
        for _ in range(repeats):
            for images, _, _ in loader:
                images = images.to(device, non_blocking=True)

                if device.type == 'cuda':
                    starter = torch.cuda.Event(enable_timing=True)
                    ender = torch.cuda.Event(enable_timing=True)
                    starter.record()
                    _ = model(images)
                    ender.record()
                    torch.cuda.synchronize()
                    elapsed_ms = starter.elapsed_time(ender)
                else:
                    start = time.perf_counter()
                    _ = model(images)
                    elapsed_ms = (time.perf_counter() - start) * 1000.0

                latencies_ms.append(elapsed_ms)

    x = np.asarray(latencies_ms, dtype=float)
    return {
        'latency_batch_size': 1,
        'latency_warmup_iterations': warmup_iterations,
        'latency_repeats': repeats,
        'latency_n_measurements': len(x),
        'latency_mean_ms': x.mean(),
        'latency_std_ms': x.std(ddof=1),
        'latency_median_ms': np.median(x),
        'latency_ci_lower_ms': np.percentile(x, 2.5),
        'latency_ci_upper_ms': np.percentile(x, 97.5),
    }

## One complete seed run

In [ ]:
def run_one_seed(seed):
    run_dir = OUTPUT_DIR / f"seed_{seed}"
    run_dir.mkdir(parents=True, exist_ok=True)

    result_path = run_dir / "run_results.csv"
    checkpoint_path = run_dir / "best_checkpoint.pth"

    # Resume safely if this seed was fully completed earlier.
    if result_path.exists():
        print(f"Seed {seed} already completed; loading saved result.")
        return pd.read_csv(result_path).iloc[0].to_dict()

    print("\n" + "=" * 90)
    print(f"Model: {CFG.MODEL_NAME} | Seed: {seed}")
    print("=" * 90)

    set_seed(seed)
    train_loader, val_loader, test_loader = create_loaders(seed)

    model = timm.create_model(
        CFG.MODEL_NAME,
        pretrained=True,
        num_classes=CFG.NUM_CLASSES,
    ).to(DEVICE)

    # Confirm that the complete pretrained model is being fine-tuned.
    if not all(parameter.requires_grad for parameter in model.parameters()):
        raise RuntimeError("Some model parameters are unexpectedly frozen.")

    class_weights = torch.tensor(
        class_weights_np,
        dtype=torch.float32,
        device=DEVICE,
    )

    criterion = nn.CrossEntropyLoss(weight=class_weights)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=6,
    )

    best_score = -np.inf
    best_epoch = None
    patience_counter = 0
    history = []

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    training_start = time.perf_counter()

    for epoch in range(1, CFG.NUM_EPOCHS + 1):
        train_loss, train_metrics, train_time = train_one_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            DEVICE,
        )

        val_loss, val_metrics, _, val_time = evaluate(
            model,
            val_loader,
            criterion,
            DEVICE,
            return_proba=False,
        )

        current_score = float(val_metrics[CFG.SELECTION_METRIC])
        scheduler.step(current_score)

        row = {
            "seed": int(seed),
            "epoch": int(epoch),
            "learning_rate": float(optimizer.param_groups[0]["lr"]),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "train_time_sec": float(train_time),
            "val_time_sec": float(val_time),
        }

        for key, value in train_metrics.items():
            row[f"train_{key}"] = float(value)

        for key, value in val_metrics.items():
            row[f"val_{key}"] = float(value)

        history.append(row)

        print(
            f"Epoch {epoch:02d}/{CFG.NUM_EPOCHS} | "
            f"Train loss {train_loss:.4f}, "
            f"macro-F1 {train_metrics['f1_macro']:.4f} | "
            f"Val loss {val_loss:.4f}, "
            f"macro-F1 {val_metrics['f1_macro']:.4f}, "
            f"QWK {val_metrics['qwk']:.4f}"
        )

        if current_score > best_score:
            best_score = current_score
            best_epoch = epoch
            patience_counter = 0

            # Convert metadata to standard Python types before saving.
            checkpoint_val_metrics = {
                key: float(value)
                for key, value in val_metrics.items()
            }

            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "seed": int(seed),
                    "epoch": int(epoch),
                    "selection_metric": str(CFG.SELECTION_METRIC),
                    "selection_score": float(current_score),
                    "val_loss": float(val_loss),
                    "val_metrics": checkpoint_val_metrics,
                    "model_identifier": str(CFG.MODEL_NAME),
                },
                checkpoint_path,
            )

        else:
            patience_counter += 1

            if patience_counter >= CFG.PATIENCE:
                print(f"Early stopping at epoch {epoch}.")
                break

    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

    total_training_time = time.perf_counter() - training_start

    peak_gpu_memory_mb = (
        torch.cuda.max_memory_allocated() / (1024**2)
        if DEVICE.type == "cuda"
        else np.nan
    )

    history_df = pd.DataFrame(history)
    history_df.to_csv(
        run_dir / "training_history.csv",
        index=False,
    )

    # Ensure that a checkpoint was successfully created.
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"No checkpoint was saved for seed {seed}: {checkpoint_path}"
        )

    # Load the best checkpoint selected using the validation metric.
    checkpoint = torch.load(
        checkpoint_path,
        map_location=DEVICE,
        weights_only=False,
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(DEVICE)
    model.eval()

    test_loss, test_metrics, test_pred_df, test_eval_time = evaluate(
        model,
        test_loader,
        criterion,
        DEVICE,
        return_proba=True,
    )

    test_pred_df.insert(0, "seed", int(seed))
    test_pred_df.insert(0, "model", CFG.SAVE_NAME)
    test_pred_df.to_csv(
        run_dir / "test_predictions.csv",
        index=False,
    )

    per_class_df = calculate_per_class_metrics(
        test_pred_df["y_true"].to_numpy(),
        test_pred_df["y_pred"].to_numpy(),
    )

    per_class_df.insert(0, "seed", int(seed))
    per_class_df.insert(0, "model", CFG.SAVE_NAME)
    per_class_df.to_csv(
        run_dir / "per_class_metrics.csv",
        index=False,
    )

    cm = confusion_matrix(
        test_pred_df["y_true"],
        test_pred_df["y_pred"],
        labels=list(range(CFG.NUM_CLASSES)),
    )

    pd.DataFrame(
        cm,
        index=CFG.CLASS_NAMES,
        columns=CFG.CLASS_NAMES,
    ).to_csv(run_dir / "confusion_matrix.csv")

    total_params = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    trainable_params = sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )

    latency = measure_batch1_forward_latency(
        model,
        create_latency_loader(),
        DEVICE,
        warmup_iterations=CFG.LATENCY_WARMUP,
        repeats=CFG.LATENCY_REPEATS,
    )

    result = {
        "model": CFG.SAVE_NAME,
        "model_identifier": CFG.MODEL_NAME,
        "seed": int(seed),
        "batch_size_training": int(CFG.BATCH_SIZE),
        "selection_metric": checkpoint["selection_metric"],
        "selected_epoch": int(checkpoint["epoch"]),
        "best_val_selection_score": float(
            checkpoint["selection_score"]
        ),
        "selected_val_loss": float(checkpoint["val_loss"]),
        "epochs_completed": int(len(history_df)),
        "early_stopping_used": bool(
            len(history_df) < CFG.NUM_EPOCHS
        ),
        "test_loss": float(test_loss),
        **{
            f"test_{key}": float(value)
            for key, value in test_metrics.items()
        },
        "total_parameters": int(total_params),
        "trainable_parameters": int(trainable_params),
        "total_parameters_million": float(
            total_params / 1e6
        ),
        "trainable_parameters_million": float(
            trainable_params / 1e6
        ),
        "total_training_time_sec": float(
            total_training_time
        ),
        "mean_train_epoch_time_sec": float(
            history_df["train_time_sec"].mean()
        ),
        "test_evaluation_time_sec": float(
            test_eval_time
        ),
        "peak_gpu_memory_mb": float(
            peak_gpu_memory_mb
        ),
        **{
            key: float(value)
            for key, value in latency.items()
        },
    }

    pd.DataFrame([result]).to_csv(
        result_path,
        index=False,
    )

    # Release memory before starting the next seed.
    del (
        model,
        optimizer,
        scheduler,
        criterion,
        train_loader,
        val_loader,
        test_loader,
    )

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## Run all five seeds

In [ ]:
all_results = []

for seed in CFG.SEEDS:
    all_results.append(run_one_seed(seed))

seed_results_df = (
    pd.DataFrame(all_results)
    .sort_values("seed")
    .reset_index(drop=True)
)

seed_results_df.to_csv(
    OUTPUT_DIR / "all_seed_results.csv",
    index=False,
)

seed_results_df


Model: mobilenetv2_100 | Seed: 42


model.safetensors: reconstructing file:   0%|          |  0.00B / 14.2MB            

model.safetensors: downloading bytes:           |  0.00B            

Epoch 01/20 | Train loss 2.1932, macro-F1 0.2917 | Val loss 2.0054, macro-F1 0.3511, QWK 0.3259
Epoch 02/20 | Train loss 1.3780, macro-F1 0.4440 | Val loss 1.5462, macro-F1 0.4555, QWK 0.6283
Epoch 03/20 | Train loss 1.0578, macro-F1 0.5274 | Val loss 1.3780, macro-F1 0.4701, QWK 0.6853
Epoch 04/20 | Train loss 0.9119, macro-F1 0.5714 | Val loss 1.3722, macro-F1 0.5078, QWK 0.7503
Epoch 05/20 | Train loss 0.8052, macro-F1 0.6068 | Val loss 1.2991, macro-F1 0.5010, QWK 0.7207
Epoch 06/20 | Train loss 0.7124, macro-F1 0.6259 | Val loss 1.1945, macro-F1 0.5240, QWK 0.7626
Epoch 07/20 | Train loss 0.6235, macro-F1 0.6490 | Val loss 1.2172, macro-F1 0.5289, QWK 0.7641
Epoch 08/20 | Train loss 0.5807, macro-F1 0.6712 | Val loss 1.2307, macro-F1 0.5118, QWK 0.7802
Epoch 09/20 | Train loss 0.5543, macro-F1 0.7017 | Val loss 1.1734, macro-F1 0.5518, QWK 0.7923
Epoch 10/20 | Train loss 0.4948, macro-F1 0.7218 | Val loss 1.0945, macro-F1 0.5929, QWK 0.8093
Epoch 11/20 | Train loss 0.4754, macro-F


Model: mobilenetv2_100 | Seed: 123
Epoch 01/20 | Train loss 2.0418, macro-F1 0.3517 | Val loss 2.0363, macro-F1 0.3764, QWK 0.6092
Epoch 02/20 | Train loss 1.2230, macro-F1 0.4873 | Val loss 1.7878, macro-F1 0.3690, QWK 0.6752
Epoch 03/20 | Train loss 0.9871, macro-F1 0.5361 | Val loss 1.4636, macro-F1 0.4421, QWK 0.6872
Epoch 04/20 | Train loss 0.8466, macro-F1 0.5877 | Val loss 1.4074, macro-F1 0.4348, QWK 0.7471
Epoch 05/20 | Train loss 0.7740, macro-F1 0.6068 | Val loss 1.2956, macro-F1 0.4549, QWK 0.7646
Epoch 06/20 | Train loss 0.7058, macro-F1 0.6235 | Val loss 1.2935, macro-F1 0.4686, QWK 0.7635
Epoch 07/20 | Train loss 0.6227, macro-F1 0.6469 | Val loss 1.2265, macro-F1 0.4909, QWK 0.7708
Epoch 08/20 | Train loss 0.5801, macro-F1 0.6848 | Val loss 1.2266, macro-F1 0.4756, QWK 0.7652
Epoch 09/20 | Train loss 0.4951, macro-F1 0.7038 | Val loss 1.1601, macro-F1 0.5408, QWK 0.8095
Epoch 10/20 | Train loss 0.5022, macro-F1 0.7166 | Val loss 1.2235, macro-F1 0.4986, QWK 0.7766
Epoc

,model,model_identifier,seed,batch_size_training,selection_metric,selected_epoch,best_val_selection_score,selected_val_loss,epochs_completed,early_stopping_used,...,peak_gpu_memory_mb,latency_batch_size,latency_warmup_iterations,latency_repeats,latency_n_measurements,latency_mean_ms,latency_std_ms,latency_median_ms,latency_ci_lower_ms,latency_ci_upper_ms
0,mobilenetv2,mobilenetv2_100,42,32,f1_macro,20,0.658811,1.140590,20,False,...,2487.196777,1.0,30.0,5.0,1830.0,6.873074,0.301140,6.889120,6.368082,7.465397
1,mobilenetv2,mobilenetv2_100,123,32,f1_macro,20,0.620055,1.125310,20,False,...,2487.103027,1.0,30.0,5.0,1830.0,6.951327,0.335506,6.900464,6.422329,7.561046
2,mobilenetv2,mobilenetv2_100,2024,32,f1_macro,20,0.647836,1.270524,20,False,...,2487.103027,1.0,30.0,5.0,1830.0,6.935330,0.356452,6.931392,6.402794,7.560430
3,mobilenetv2,mobilenetv2_100,3407,32,f1_macro,14,0.648212,1.183311,19,True,...,2487.103027,1.0,30.0,5.0,1830.0,7.034624,0.323848,7.027072,6.485178,7.651005
4,mobilenetv2,mobilenetv2_100,7777,32,f1_macro,20,0.598464,1.179620,20,False,...,2487.103027,1.0,30.0,5.0,1830.0,7.331792,0.385952,7.280992,6.663236,7.965535


## Mean, sample standard deviation, and 95% CI across seeds

In [ ]:
REPORT_METRICS = [
    'test_accuracy',
    'test_precision_macro',
    'test_recall_macro',
    'test_f1_macro',
    'test_precision_weighted',
    'test_recall_weighted',
    'test_f1_weighted',
    'test_auc_macro_ovr',
    'test_auc_weighted_ovr',
    'test_qwk',
    'selected_epoch',
    'latency_mean_ms',
]

summary_rows = []
for metric in REPORT_METRICS:
    values = pd.to_numeric(seed_results_df[metric], errors='coerce').dropna().to_numpy(float)
    n = len(values)
    mean = values.mean()
    sd = values.std(ddof=1) if n > 1 else np.nan
    sem = sd / math.sqrt(n) if n > 1 else np.nan
    margin = t.ppf(0.975, df=n-1) * sem if n > 1 else np.nan
    summary_rows.append({
        'model': CFG.SAVE_NAME,
        'metric': metric,
        'n_runs': n,
        'mean': mean,
        'sample_sd': sd,
        'ci95_lower_across_seeds': mean - margin,
        'ci95_upper_across_seeds': mean + margin,
        'formatted_mean_sd': f'{mean:.3f} ± {sd:.3f}' if np.isfinite(sd) else f'{mean:.3f}',
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / 'mean_sd_and_seed_ci_summary.csv', index=False)
summary_df

,model,metric,n_runs,mean,sample_sd,ci95_lower_across_seeds,ci95_upper_across_seeds,formatted_mean_sd
0,mobilenetv2,test_accuracy,5,0.737158,0.008937,0.726061,0.748256,0.737 ± 0.009
1,mobilenetv2,test_precision_macro,5,0.564949,0.015234,0.546033,0.583864,0.565 ± 0.015
2,mobilenetv2,test_recall_macro,5,0.601348,0.019347,0.577325,0.625371,0.601 ± 0.019
3,mobilenetv2,test_f1_macro,5,0.568262,0.019057,0.544599,0.591924,0.568 ± 0.019
4,mobilenetv2,test_precision_weighted,5,0.779752,0.005641,0.772747,0.786756,0.780 ± 0.006
5,mobilenetv2,test_recall_weighted,5,0.737158,0.008937,0.726061,0.748256,0.737 ± 0.009
6,mobilenetv2,test_f1_weighted,5,0.750517,0.007155,0.741632,0.759401,0.751 ± 0.007
7,mobilenetv2,test_auc_macro_ovr,5,0.904052,0.008549,0.893437,0.914667,0.904 ± 0.009
8,mobilenetv2,test_auc_weighted_ovr,5,0.941260,0.004489,0.935686,0.946833,0.941 ± 0.004
9,mobilenetv2,test_qwk,5,0.816819,0.014827,0.798409,0.835229,0.817 ± 0.015


## Hierarchical stratified-bootstrap confidence intervals

This bootstrap samples both sources of uncertainty: it randomly selects one of the five seed runs and then resamples test images within each true class. It is especially useful for the small Severe and Proliferative DR classes.

In [ ]:
def load_all_prediction_runs(model_dir):
    frames = []
    for seed in CFG.SEEDS:
        path = Path(model_dir) / f'seed_{seed}' / 'test_predictions.csv'
        if not path.exists():
            raise FileNotFoundError(path)
        frames.append(pd.read_csv(path).sort_values('sample_index').reset_index(drop=True))

    reference = frames[0][['sample_index', 'y_true']]
    for df in frames[1:]:
        if not reference.equals(df[['sample_index', 'y_true']]):
            raise ValueError('Test sample order or labels differ across seed runs.')
    return frames


def hierarchical_bootstrap_over_seeds(prediction_runs, n_bootstrap=2000, seed=2026):
    rng = np.random.default_rng(seed)
    y_true = prediction_runs[0]['y_true'].to_numpy(int)
    class_indices = {c: np.where(y_true == c)[0] for c in range(CFG.NUM_CLASSES)}

    overall_records = []
    per_class_records = []

    for b in range(n_bootstrap):
        run_df = prediction_runs[rng.integers(0, len(prediction_runs))]
        sampled_indices = np.concatenate([
            rng.choice(idx, size=len(idx), replace=True)
            for idx in class_indices.values()
        ])
        rng.shuffle(sampled_indices)

        yt = run_df['y_true'].to_numpy(int)[sampled_indices]
        yp = run_df['y_pred'].to_numpy(int)[sampled_indices]
        prob = run_df[[f'prob_{c}' for c in range(CFG.NUM_CLASSES)]].to_numpy(float)[sampled_indices]

        m = calculate_metrics(yt, yp, prob)
        overall_records.append({'bootstrap': b, **m})

        pc = calculate_per_class_metrics(yt, yp)
        pc['bootstrap'] = b
        per_class_records.append(pc)

    return pd.DataFrame(overall_records), pd.concat(per_class_records, ignore_index=True)


def percentile_ci_summary(df, metrics, group_cols=None, confidence=0.95):
    alpha = 1 - confidence
    if group_cols:
        grouped = df.groupby(group_cols, dropna=False)
    else:
        grouped = [((), df)]

    rows = []
    for group_key, group in grouped:
        if not isinstance(group_key, tuple):
            group_key = (group_key,)
        base = dict(zip(group_cols or [], group_key))
        for metric in metrics:
            x = pd.to_numeric(group[metric], errors='coerce').dropna().to_numpy(float)
            rows.append({
                **base,
                'metric': metric,
                'bootstrap_mean': x.mean(),
                'ci95_lower': np.quantile(x, alpha / 2),
                'ci95_upper': np.quantile(x, 1 - alpha / 2),
                'n_bootstrap': len(x),
            })
    return pd.DataFrame(rows)

prediction_runs = load_all_prediction_runs(OUTPUT_DIR)
bootstrap_overall_df, bootstrap_per_class_df = hierarchical_bootstrap_over_seeds(
    prediction_runs,
    n_bootstrap=CFG.N_BOOTSTRAP,
)

bootstrap_overall_df.to_csv(OUTPUT_DIR / 'bootstrap_overall_distributions.csv', index=False)
bootstrap_per_class_df.to_csv(OUTPUT_DIR / 'bootstrap_per_class_distributions.csv', index=False)

overall_ci_df = percentile_ci_summary(
    bootstrap_overall_df,
    metrics=[
        'accuracy', 'precision_macro', 'recall_macro', 'f1_macro',
        'precision_weighted', 'recall_weighted', 'f1_weighted',
        'auc_macro_ovr', 'auc_weighted_ovr', 'qwk',
    ],
)
per_class_ci_df = percentile_ci_summary(
    bootstrap_per_class_df,
    metrics=['precision', 'recall', 'f1'],
    group_cols=['class_index', 'class_name'],
)

overall_ci_df.to_csv(OUTPUT_DIR / 'bootstrap_overall_ci95.csv', index=False)
per_class_ci_df.to_csv(OUTPUT_DIR / 'bootstrap_per_class_ci95.csv', index=False)

display(overall_ci_df)
display(per_class_ci_df)

,metric,bootstrap_mean,ci95_lower,ci95_upper,n_bootstrap
0,accuracy,0.738149,0.696721,0.781421,2000
1,precision_macro,0.566936,0.502160,0.638437,2000
2,recall_macro,0.601727,0.528473,0.680050,2000
3,f1_macro,0.568169,0.499944,0.643658,2000
4,precision_weighted,0.780879,0.744789,0.817089,2000
5,recall_weighted,0.738149,0.696721,0.781421,2000
6,f1_weighted,0.750906,0.714071,0.789730,2000
7,auc_macro_ovr,0.903846,0.873270,0.932778,2000
8,auc_weighted_ovr,0.941314,0.923725,0.957550,2000
9,qwk,0.817435,0.755080,0.865501,2000


,class_index,class_name,metric,bootstrap_mean,ci95_lower,ci95_upper,n_bootstrap
0,0,No DR,precision,0.978629,0.954768,0.994682,2000
1,0,No DR,recall,0.906796,0.849246,0.949749,2000
2,0,No DR,f1,0.941117,0.905660,0.969231,2000
3,1,Mild,precision,0.362888,0.259740,0.486504,2000
4,1,Mild,recall,0.693900,0.533333,0.866667,2000
5,1,Mild,f1,0.473708,0.361696,0.595238,2000
6,2,Moderate,precision,0.665130,0.573127,0.772749,2000
7,2,Moderate,recall,0.530126,0.413793,0.655460,2000
8,2,Moderate,f1,0.587721,0.489781,0.680304,2000
9,3,Severe,precision,0.313955,0.133261,0.526316,2000


In [ ]:
import shutil

shutil.make_archive(
    "/content/drive/MyDrive/DR_revision_results/mobilenetv2",      # creates /content/mobilenetv2.zip
    "zip",
    "/content/drive/MyDrive/DR_revision_results/mobilenetv2"
)

'/content/drive/MyDrive/DR_revision_results/mobilenetv2.zip'

## Aggregate per-class point estimates across seeds

In [ ]:
per_class_files = [OUTPUT_DIR / f'seed_{s}' / 'per_class_metrics.csv' for s in CFG.SEEDS]
per_class_all = pd.concat([pd.read_csv(p) for p in per_class_files], ignore_index=True)

per_class_seed_summary = (
    per_class_all
    .groupby(['class_index', 'class_name'])[['precision', 'recall', 'f1']]
    .agg(['mean', 'std'])
    .reset_index()
)
per_class_seed_summary.to_csv(OUTPUT_DIR / 'per_class_mean_sd_across_seeds.csv', index=False)
per_class_seed_summary

class_index        class_name precision              recall            \
                                     mean       std      mean       std   
0           0             No DR  0.978294  0.003717  0.905528  0.018936   
1           1              Mild  0.357828  0.039887  0.693333  0.027889   
2           2          Moderate  0.664023  0.027853  0.528736  0.043769   
3           3            Severe  0.313000  0.055152  0.388235  0.052613   
4           4  Proliferative DR  0.511598  0.058696  0.490909  0.049793   

         f1            
       mean       std  
0  0.940421  0.010490  
1  0.470596  0.033809  
2  0.587593  0.026445  
3  0.343885  0.044403  
4  0.498813  0.039599

## Environment and reproducibility record

In [ ]:
environment = {
    'python': platform.python_version(),
    'pytorch': torch.__version__,
    'torchvision': __import__('torchvision').__version__,
    'timm': timm.__version__,
    'cuda_available': torch.cuda.is_available(),
    'cuda_version': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'model_identifier': CFG.MODEL_NAME,
    'training_batch_size': CFG.BATCH_SIZE,
    'latency_batch_size': 1,
    'latency_definition': 'forward pass only; excludes disk loading and host-to-device transfer',
    'seeds': CFG.SEEDS,
    'selection_metric': CFG.SELECTION_METRIC,
}
with open(OUTPUT_DIR / 'environment.json', 'w') as f:
    json.dump(environment, f, indent=2)
print(json.dumps(environment, indent=2))

{
  "python": "3.12.13",
  "pytorch": "2.11.0+cu128",
  "torchvision": "0.26.0+cu128",
  "timm": "1.0.28",
  "cuda_available": true,
  "cuda_version": "12.8",
  "gpu": "NVIDIA A100-SXM4-80GB",
  "model_identifier": "mobilenetv2_100",
  "training_batch_size": 32,
  "latency_batch_size": 1,
  "latency_definition": "forward pass only; excludes disk loading and host-to-device transfer",
  "seeds": [
    42,
    123,
    2024,
    3407,
    7777
  ],
  "selection_metric": "f1_macro"
}


# Cross-model statistical analysis

Run the cells below **after completing all six architectures**. Put every model folder under the same `CFG.OUTPUT_ROOT`. The code performs:

1. Friedman omnibus tests on repeated-run metrics;
2. paired Wilcoxon signed-rank tests with Holm correction;
3. exact McNemar tests for matched-seed predictions;
4. paired stratified-bootstrap tests for multiclass macro AUC, macro F1, QWK, and accuracy.

With only five seeds, statistical power is limited. Report non-significant differences honestly.

In [ ]:
MODEL_FOLDERS = [
    'resnet50',
    'efficientnet_b0',
    'mobilenetv2',
    'vit_base',
    'swin_tiny',
    'swin_base',
]


def load_all_model_seed_results(root, model_folders):
    frames = []
    for model in model_folders:
        p = Path(root) / model / 'all_seed_results.csv'
        if p.exists():
            frames.append(pd.read_csv(p))
        else:
            print('Missing:', p)
    if not frames:
        raise FileNotFoundError('No all_seed_results.csv files found.')
    return pd.concat(frames, ignore_index=True)

# Uncomment after all models finish:
# all_models_df = load_all_model_seed_results(CFG.OUTPUT_ROOT, MODEL_FOLDERS)
# all_models_df.to_csv(Path(CFG.OUTPUT_ROOT) / 'all_models_all_seeds.csv', index=False)
# all_models_df.head()

In [ ]:
def friedman_and_wilcoxon_holm(all_models_df, metrics):
    omnibus_rows = []
    posthoc_rows = []

    for metric in metrics:
        pivot = all_models_df.pivot(index='seed', columns='model', values=metric).dropna()
        if pivot.shape[0] < 3 or pivot.shape[1] < 3:
            print(f'Skipping {metric}: insufficient complete data.')
            continue

        statistic, p_value = friedmanchisquare(*[pivot[c].to_numpy() for c in pivot.columns])
        omnibus_rows.append({
            'metric': metric,
            'friedman_statistic': statistic,
            'p_value': p_value,
            'n_seeds': len(pivot),
            'n_models': len(pivot.columns),
        })

        raw_rows = []
        for a, b in combinations(pivot.columns, 2):
            xa, xb = pivot[a].to_numpy(), pivot[b].to_numpy()
            try:
                stat, p = wilcoxon(xa, xb, alternative='two-sided', zero_method='wilcox')
            except ValueError:
                stat, p = np.nan, 1.0
            raw_rows.append({
                'metric': metric,
                'model_a': a,
                'model_b': b,
                'mean_a': xa.mean(),
                'mean_b': xb.mean(),
                'wilcoxon_statistic': stat,
                'raw_p': p,
            })

        temp = pd.DataFrame(raw_rows)
        reject, corrected, _, _ = multipletests(temp['raw_p'], alpha=0.05, method='holm')
        temp['holm_adjusted_p'] = corrected
        temp['significant_after_holm'] = reject
        posthoc_rows.append(temp)

    omnibus_df = pd.DataFrame(omnibus_rows)
    posthoc_df = pd.concat(posthoc_rows, ignore_index=True) if posthoc_rows else pd.DataFrame()
    return omnibus_df, posthoc_df

# Example after loading all_models_df:
omnibus_df, posthoc_df = friedman_and_wilcoxon_holm(
    all_models_df,
    metrics=['test_accuracy', 'test_f1_macro', 'test_qwk']
)
omnibus_df.to_csv(Path(CFG.OUTPUT_ROOT) / 'friedman_tests.csv', index=False)
posthoc_df.to_csv(Path(CFG.OUTPUT_ROOT) / 'wilcoxon_holm_posthoc.csv', index=False)

Skipping test_accuracy: insufficient complete data.
Skipping test_f1_macro: insufficient complete data.
Skipping test_qwk: insufficient complete data.


In [ ]:
def load_prediction_file(root, model, seed):
    return pd.read_csv(Path(root) / model / f'seed_{seed}' / 'test_predictions.csv').sort_values('sample_index')


def exact_mcnemar_for_models(root, model_a, model_b, seeds):
    rows = []
    for seed in seeds:
        a = load_prediction_file(root, model_a, seed)
        b = load_prediction_file(root, model_b, seed)
        if not a[['sample_index', 'y_true']].equals(b[['sample_index', 'y_true']]):
            raise ValueError(f'Sample mismatch for seed {seed}: {model_a} vs {model_b}')

        y = a['y_true'].to_numpy()
        ca = a['y_pred'].to_numpy() == y
        cb = b['y_pred'].to_numpy() == y
        table = np.array([
            [np.sum(ca & cb), np.sum(ca & ~cb)],
            [np.sum(~ca & cb), np.sum(~ca & ~cb)],
        ])
        test = mcnemar(table, exact=True)
        rows.append({
            'seed': seed,
            'model_a': model_a,
            'model_b': model_b,
            'both_correct': table[0, 0],
            'a_correct_b_wrong': table[0, 1],
            'a_wrong_b_correct': table[1, 0],
            'both_wrong': table[1, 1],
            'mcnemar_p': test.pvalue,
        })
    out = pd.DataFrame(rows)
    reject, adjusted, _, _ = multipletests(out['mcnemar_p'], alpha=0.05, method='holm')
    out['holm_adjusted_p_across_seeds'] = adjusted
    out['significant_after_holm'] = reject
    return out

# Pre-specify only scientifically important comparisons, e.g.:
# exact_mcnemar_for_models(CFG.OUTPUT_ROOT, 'efficientnet_b0', 'swin_base', CFG.SEEDS)
# exact_mcnemar_for_models(CFG.OUTPUT_ROOT, 'vit_base', 'swin_base', CFG.SEEDS)